In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

matches_df = pd.read_csv("Data/Raw/charting-m-matches.csv")
m_points_2020 = pd.read_csv("Data/Raw/charting-m-points-2020s.csv")
m_points_2010 = pd.read_csv("Data/Raw/charting-m-points-2010s.csv")
m_points_2009 = pd.read_csv("Data/Raw/charting-m-points-to-2009.csv")

matches = matches_df[["match_id", "Date", "Tournament", "Surface"]]
points_2020 = m_points_2020[["match_id", "Set1", "Set2", "Gm1", "Gm2",
                            "Pts", "Gm#", "TbSet", "Svr", "PtWinner"]]
points_2010 = m_points_2010[["match_id", "Set1", "Set2", "Gm1", "Gm2",
                            "Pts", "Gm#", "TbSet", "Svr", "PtWinner"]]
points_2009 = m_points_2009[["match_id", "Set1", "Set2", "Gm1", "Gm2",
                            "Pts", "Gm#", "TbSet", "Svr", "PtWinner"]]

points_df = pd.concat([points_2020, points_2010, points_2009 ],ignore_index=True)

final_df = points_df.merge(matches, on="match_id", how="left")
final_df["Date"] = pd.to_datetime(final_df["Date"], errors="coerce", format="%Y%m%d")

final_df = final_df[final_df["Date"].notna()]
final_df = final_df[final_df["Date"] >= "2000-01-01"]

final_df = final_df.dropna()

Categorical_data = ["Surface"]
Numerical_data = ["Set1", "Set2", "Gm1", "Gm2", "Gm#", "TbSet"]

y = final_df["PtWinner"]
X = final_df[ Numerical_data + Categorical_data]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers= [(
        "cat",
        OneHotEncoder(handle_unknown="ignore"),
        Categorical_data
    )],
    remainder="passthrough"  
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

model.fit(X_train, y_train)

scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")

print("CV Accuracy:", scores)
print("Mean", scores.mean())
print("Std:", scores.std())


C:\Users\danie\AppData\Local\Temp\ipykernel_36140\791394722.py:11: DtypeWarning: Columns (0: TbSet) have mixed types. Specify dtype option on import or set low_memory=False.
  m_points_2020 = pd.read_csv("Data/Raw/charting-m-points-2020s.csv")


CV Accuracy: [0.63168078 0.63105738 0.62988446 0.63014597 0.62301607]
Mean 0.6291569337551313
Std: 0.0031368502448273685
